In [1]:
import os, sys
print("cwd:", os.getcwd())
print("repo cloned:", os.path.isdir("/content/ML_AL_MC"))
print("src exists :", os.path.isdir("/content/ML_AL_MC/src"))
print("on sys.path:", [p for p in sys.path if "ML_AL_MC" in p])

cwd: /content
repo cloned: False
src exists : False
on sys.path: []


In [2]:
from google.colab import drive, userdata
import os, sys
drive.mount('/content/drive')

TOKEN = userdata.get('GITHUB_TOKEN')
USER, REPO = "jeevithrao", "ML_AL_MC"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/{USER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git remote set-url origin https://{TOKEN}@github.com/{USER}/{REPO}.git
!git config user.name "jeevithrao"
!git config user.email "138958136+jeevithrao@users.noreply.github.com"
!git pull origin main
sys.path.append(f"{REPO_DIR}/src")
os.environ["PYTHONPATH"] = f"{REPO_DIR}/src"
print("setup complete")

Mounted at /content/drive
Cloning into '/content/ML_AL_MC'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 75 (delta 16), reused 70 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 2.65 MiB | 20.68 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/ML_AL_MC
From https://github.com/jeevithrao/ML_AL_MC
 * branch            main       -> FETCH_HEAD
Already up to date.
setup complete


In [3]:
import os, sys, json
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
assert any("ML_AL_MC/src" in p for p in sys.path), "Run the setup cell (Block 4.1) first"
!pip install -q timm scikit-learn

TAR = "/content/drive/MyDrive/birdclef_project/mels.tar"
assert os.path.exists(TAR), "mels.tar missing on Drive"
if not os.path.exists("/content/data/mels"):
    !mkdir -p /content/data
    !tar -xf {TAR} -C /content/data

from data_prep import prepare
from data_utils import MelDataset, sqrt_inv_weights
from engine import DEV, make_loader, predict
from metrics import macro_auc
from model import make_model
from train_config import *

CK = "/content/drive/MyDrive/birdclef_project/checkpoints"
idx, Y, rows, CLASSES, ALL = prepare()
w = sqrt_inv_weights(idx.primary_label.values[rows["train"]])
train_ds, val_ds, test_ds = (MelDataset(ALL, rows[s], Y) for s in ["train", "val", "test"])
print("ready:", {s: len(r) for s, r in rows.items()})

  0%|          | 0/24459 [00:00<?, ?it/s]

ready: {'train': 45946, 'val': 9843, 'test': 9855}


In [4]:
%%writefile src/profile_tools.py
import copy, io, os, time, numpy as np, torch
from torch.utils.flop_counter import FlopCounterMode

def count_params(m):
    return sum(p.numel() for p in m.parameters())

def size_mb(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

def gflops(m, shape=(1, 1, 128, 313)):
    with FlopCounterMode(display=False) as fc, torch.no_grad():
        m(torch.randn(*shape))
    return fc.get_total_flops() / 1e9

@torch.no_grad()
def cpu_latency_ms(m, batch, runs, warmup=10):
    x = torch.randn(batch, 1, 128, 313)
    for _ in range(warmup): m(x)
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); m(x); ts.append(time.perf_counter() - t0)
    return float(np.median(ts) * 1000)

def mel_ms(n=30):
    import librosa
    from audio_config import SR, CHUNK_SEC, N_FFT, HOP, N_MELS, FMIN, FMAX, TOP_DB
    y = (np.random.randn(SR * CHUNK_SEC) * 0.1).astype(np.float32)
    def one():
        m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                           n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
        return librosa.power_to_db(m, ref=np.max, top_db=TOP_DB)
    for _ in range(3): one()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); one(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts) * 1000)

def profile(name, model, extra=None, threads=2):
    m = copy.deepcopy(model).eval().cpu()
    torch.set_num_threads(threads)
    mel, l1, l32 = mel_ms(), cpu_latency_ms(m, 1, 100), cpu_latency_ms(m, 32, 20)
    row = dict(model=name, params_M=round(count_params(m) / 1e6, 3), size_MB=round(size_mb(m), 2),
               gflops=round(gflops(m), 3), lat_b1_ms=round(l1, 1), lat_b32_ms=round(l32, 1),
               mel_ms=round(mel, 1), threads=threads,
               win_per_s_b1=round(1000 / (mel + l1), 1),
               win_per_s_b32=round(32 * 1000 / (32 * mel + l32), 1))
    if extra: row.update(extra)
    return row

def log_row(row, path="results/efficiency.csv"):
    import pandas as pd
    df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    if len(df): df = df[df.model != row["model"]]
    pd.concat([df, pd.DataFrame([row])]).to_csv(path, index=False)

Overwriting src/profile_tools.py


In [5]:
from profile_tools import profile, log_row
base = json.load(open("results/baseline_metrics.json"))

teacher = make_model("efficientnet_b0", pretrained=False).to(DEV)
teacher.load_state_dict(torch.load(f"{CK}/baseline_best.pt", map_location=DEV)); teacher.eval()

def evaluate(m):
    va, _ = macro_auc(Y[rows["val"]], predict(m, make_loader(val_ds, False)))
    te, _ = macro_auc(Y[rows["test"]], predict(m, make_loader(test_ds, False)))
    return round(va, 4), round(te, 4)

va, te = evaluate(teacher)
print(f"teacher reproduced: val {va} | test {te} | saved: val {base['val_auc']:.4f} test {base['test_auc']:.4f}")
row = profile("B0 baseline FP32", teacher, dict(val_auc=va, test_auc=te))
log_row(row); print(row)

teacher reproduced: val 0.9627 | test 0.9653 | saved: val 0.9627 test 0.9653
{'model': 'B0 baseline FP32', 'params_M': 4.24, 'size_MB': 17.25, 'gflops': 0.615, 'lat_b1_ms': 38.4, 'lat_b32_ms': 1390.9, 'mel_ms': 7.7, 'threads': 2, 'win_per_s_b1': 21.7, 'win_per_s_b32': 19.5, 'val_auc': 0.9627, 'test_auc': 0.9653}


In [6]:
%%writefile src/distill.py
import os, gc, time, numpy as np, torch, torch.nn as nn
import torch.nn.functional as Fn
from engine import DEV, prep, time_shift, spec_augment, mixup, make_loader, predict
from train_config import *
from metrics import macro_auc

def kd_loss(zs, zt, y, T, alpha):
    hard = Fn.binary_cross_entropy_with_logits(zs, y)
    if alpha >= 1.0:
        return hard
    soft = Fn.binary_cross_entropy_with_logits(zs / T, torch.sigmoid(zt / T))
    return alpha * hard + (1 - alpha) * (T * T) * soft

def fit_kd(student, teacher, train_ds, val_ds, Y, weights, T=2.0, alpha=0.5,
           epochs=EPOCHS, lr=LR, ckpt_dir=None, tag="kd", resume=True):
    use_kd = alpha < 1.0
    if use_kd: teacher.eval()
    tl, vl = make_loader(train_ds, True, weights), make_loader(val_ds, False)
    Yv = Y[val_ds.rows]
    opt = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(tl), pct_start=PCT_START)
    scaler = torch.amp.GradScaler("cuda")
    hist, best, start = [], -1.0, 0
    last = f"{ckpt_dir}/{tag}_last.pt" if ckpt_dir else None
    if last and resume and os.path.exists(last):
        s = torch.load(last, map_location=DEV, weights_only=False)
        student.load_state_dict(s["model"]); opt.load_state_dict(s["opt"])
        sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"])
        hist, best, start = s["hist"], s["best"], s["epoch"] + 1
        print(f"resumed from epoch {start}")
    for ep in range(start, epochs):
        student.train(); t0 = time.time(); tot, n = 0.0, 0
        for x, y, _ in tl:
            x = prep(x.to(DEV, non_blocking=True)); y = y.to(DEV, non_blocking=True)
            x = spec_augment(time_shift(x))
            if np.random.rand() < MIXUP_P:
                x, y = mixup(x, y)
            zt = None
            if use_kd:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                    zt = teacher(x).float()
            with torch.autocast("cuda", dtype=torch.float16):
                zs = student(x)
            loss = kd_loss(zs.float(), zt, y, T, alpha)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(student.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(x); n += len(x)
        auc, k = macro_auc(Yv, predict(student, vl))
        hist.append(dict(epoch=ep, loss=tot / n, val_auc=auc, classes=k, sec=time.time() - t0))
        print(f"[{tag}] ep {ep+1}/{epochs} | loss {tot/n:.4f} | val macro-AUC {auc:.4f} | {time.time()-t0:.0f}s")
        if ckpt_dir:
            if auc > best:
                best = auc; torch.save(student.state_dict(), f"{ckpt_dir}/{tag}_best.pt")
            torch.save(dict(model=student.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                            scaler=scaler.state_dict(), hist=hist, best=best, epoch=ep), last)
    del tl, vl; gc.collect()
    return hist

Overwriting src/distill.py


In [7]:
!git add src/ && git commit -qm "Phase 5: profiling and distillation modules" && git push -q
from distill import kd_loss
stu0 = make_model("mobilenetv3_small_100", pretrained=False)
r = profile("mnv3_small (untrained)", stu0)
print(pd.DataFrame([row, r])[["model", "params_M", "size_MB", "gflops", "lat_b1_ms", "lat_b32_ms", "mel_ms", "win_per_s_b1"]].to_string(index=False))

x, y, _ = next(iter(make_loader(train_ds, True, w)))
from engine import prep
with torch.no_grad():
    zt = teacher(prep(x.to(DEV)))
    zs = stu0.to(DEV)(prep(x.to(DEV)))
print("shapes:", tuple(zs.shape), "| kd loss finite:", bool(torch.isfinite(kd_loss(zs, zt, y.to(DEV), 2.0, 0.5))))

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	results/efficiency.csv

nothing added to commit but untracked files present (use "git add" to track)
                 model  params_M  size_MB  gflops  lat_b1_ms  lat_b32_ms  mel_ms  win_per_s_b1
      B0 baseline FP32     4.240    17.25   0.615       38.4      1390.9     7.7          21.7
mnv3_small (untrained)     1.704     6.95   0.085       13.5       172.8    27.9          24.2
shapes: (64, 182) | kd loss finite: True


In [8]:
runs = {"mnv3s_ce":        dict(alpha=1.0, T=1.0),
        "mnv3s_kd_T2_a05": dict(alpha=0.5, T=2.0)}
from distill import fit_kd
for tag, cfg in runs.items():
    torch.manual_seed(SEED); np.random.seed(SEED)
    stu = make_model("mobilenetv3_small_100").to(DEV)
    hist = fit_kd(stu, teacher, train_ds, val_ds, Y, w, ckpt_dir=CK, tag=tag, **cfg)
    stu.load_state_dict(torch.load(f"{CK}/{tag}_best.pt", map_location=DEV))
    va, te = evaluate(stu)
    log_row(profile(tag, stu, dict(val_auc=va, test_auc=te, **cfg)))
    pd.DataFrame(hist).to_csv(f"results/{tag}_history.csv", index=False)
    print(f"== {tag}: val {va} | test {te}")
    del stu; torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2MB            

model.safetensors: downloading bytes:           |  0.00B            

resumed from epoch 10
== mnv3s_ce: val 0.9567 | test 0.9556
resumed from epoch 10
== mnv3s_kd_T2_a05: val 0.9611 | test 0.9634


In [9]:
eff = pd.read_csv("results/efficiency.csv")
cols = ["model", "params_M", "size_MB", "gflops", "lat_b1_ms", "lat_b32_ms", "mel_ms",
        "win_per_s_b1", "win_per_s_b32", "val_auc", "test_auc"]
print(eff[[c for c in cols if c in eff.columns]].to_string(index=False))
b = eff[eff.model.str.startswith("B0")].iloc[0]
for _, r in eff.iterrows():
    print(f"{r.model}: {b.params_M/r.params_M:.1f}x fewer params | {b.size_MB/r.size_MB:.1f}x smaller | "
          f"{b.lat_b1_ms/r.lat_b1_ms:.1f}x faster (batch 1) | {r.win_per_s_b1:.1f} windows/s")

           model  params_M  size_MB  gflops  lat_b1_ms  lat_b32_ms  mel_ms  win_per_s_b1  win_per_s_b32  val_auc  test_auc
B0 baseline FP32     4.240    17.25   0.615       38.4      1390.9     7.7          21.7           19.5   0.9627    0.9653
        mnv3s_ce     1.704     6.95   0.085       14.1       556.4    15.1          34.2           30.7   0.9567    0.9556
 mnv3s_kd_T2_a05     1.704     6.95   0.085       13.7       199.5     7.7          46.6           71.5   0.9611    0.9634
B0 baseline FP32: 1.0x fewer params | 1.0x smaller | 1.0x faster (batch 1) | 21.7 windows/s
mnv3s_ce: 2.5x fewer params | 2.5x smaller | 2.7x faster (batch 1) | 34.2 windows/s
mnv3s_kd_T2_a05: 2.5x fewer params | 2.5x smaller | 2.8x faster (batch 1) | 46.6 windows/s


In [10]:
from distill import fit_kd
cfgs = {"mnv3s_ce": dict(alpha=1.0, T=1.0), "mnv3s_kd_T2_a05": dict(alpha=0.5, T=2.0)}
res = []
for seed in [1]:
    for tag, cfg in cfgs.items():
        torch.manual_seed(seed); np.random.seed(seed)
        stu = make_model("mobilenetv3_small_100").to(DEV)
        fit_kd(stu, teacher, train_ds, val_ds, Y, w, ckpt_dir=CK, tag=f"{tag}_s{seed}", **cfg)
        stu.load_state_dict(torch.load(f"{CK}/{tag}_s{seed}_best.pt", map_location=DEV))
        va, te = evaluate(stu)
        res.append(dict(model=tag, seed=seed, val_auc=va, test_auc=te))
        del stu; torch.cuda.empty_cache()

eff = pd.read_csv("results/efficiency.csv")
s0 = eff[eff.model.isin(cfgs)][["model", "val_auc", "test_auc"]].assign(seed=0)
D = pd.concat([s0, pd.DataFrame(res)]); D.to_csv("results/distill_seeds.csv", index=False)
piv = D.pivot(index="seed", columns="model", values=["val_auc", "test_auc"])
print(piv.round(4))
for m in ["val_auc", "test_auc"]:
    d = piv[m]["mnv3s_kd_T2_a05"] - piv[m]["mnv3s_ce"]
    print(m, "distilled minus control per seed:", d.round(4).tolist(), "| mean", round(float(d.mean()), 4))

resumed from epoch 10
resumed from epoch 10
       val_auc                 test_auc                
model mnv3s_ce mnv3s_kd_T2_a05 mnv3s_ce mnv3s_kd_T2_a05
seed                                                   
0       0.9567          0.9611   0.9556          0.9634
1       0.9466          0.9587   0.9566          0.9599
val_auc distilled minus control per seed: [0.0044, 0.0121] | mean 0.0082
test_auc distilled minus control per seed: [0.0078, 0.0033] | mean 0.0055


In [11]:
from model import make_model
!pip install -q onnx onnxruntime
import onnx, onnxruntime as ort, time
TAG = "mnv3s_kd_T2_a05"
student = make_model("mobilenetv3_small_100", pretrained=False)
student.load_state_dict(torch.load(f"{CK}/{TAG}_best.pt", map_location="cpu")); student.eval()

os.makedirs("/content/export", exist_ok=True)
FP32 = "/content/export/student_fp32.onnx"
torch.onnx.export(student, torch.randn(1, 1, 128, 313), FP32,
                  input_names=["mel"], output_names=["logits"],
                  dynamic_axes={"mel": {0: "batch"}, "logits": {0: "batch"}},
                  opset_version=17, dynamo=False)
onnx.checker.check_model(onnx.load(FP32))

x = torch.randn(8, 1, 128, 313)
with torch.no_grad(): ref = student(x).numpy()
out = ort.InferenceSession(FP32, providers=["CPUExecutionProvider"]).run(None, {"mel": x.numpy()})[0]
print("max |torch - onnx| on logits:", float(np.abs(ref - out).max()))
print("fp32 onnx size (MB):", round(os.path.getsize(FP32) / 1e6, 2))

/tmp/ipykernel_2103/3488011365.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(student, torch.randn(1, 1, 128, 313), FP32,


max |torch - onnx| on logits: 8.58306884765625e-06
fp32 onnx size (MB): 6.82


In [12]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
from onnxruntime.quantization.shape_inference import quant_pre_process
from engine import prep

PRE, INT8 = "/content/export/student_pre.onnx", "/content/export/student_int8.onnx"
try:
    quant_pre_process(FP32, PRE); src = PRE
except Exception as e:
    print("pre-processing skipped:", e); src = FP32

rng = np.random.RandomState(0)
cal_rows = rng.choice(len(rows["train"]), 320, replace=False, p=w / w.sum())
mm = np.load(ALL, mmap_mode="r")
batches = []
for s in range(0, len(cal_rows), 16):
    xb = torch.from_numpy(np.stack([np.array(mm[rows["train"][i]], dtype=np.float32) for i in cal_rows[s:s+16]]))
    batches.append({"mel": prep(xb).numpy()})

class Reader(CalibrationDataReader):
    def __init__(self, b): self.it = iter(b)
    def get_next(self): return next(self.it, None)

quantize_static(src, INT8, Reader(batches), quant_format=QuantFormat.QDQ,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, per_channel=True)
print("int8 size (MB):", round(os.path.getsize(INT8) / 1e6, 2))

int8 size (MB): 2.04


In [13]:
import copy, time
def robust_ms(fn, blocks=5, runs=40, warmup=10):
    for _ in range(warmup): fn()
    meds = []
    for _ in range(blocks):
        ts = []
        for _ in range(runs):
            t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
        meds.append(np.median(ts) * 1000)
    return round(min(meds), 1), round(float(np.median(meds)), 1), round(max(meds), 1)

def torch_fn(model, batch):
    m = copy.deepcopy(model).eval().cpu()
    x = torch.randn(batch, 1, 128, 313)
    def f():
        with torch.no_grad(): m(x)
    return f

def load_student(tag):
    m = make_model("mobilenetv3_small_100", pretrained=False)
    m.load_state_dict(torch.load(f"{CK}/{tag}_best.pt", map_location="cpu")); return m

torch.set_num_threads(2)
models = {"B0 baseline FP32": teacher, "mnv3s_ce": load_student("mnv3s_ce"),
          "mnv3s_kd_T2_a05": load_student("mnv3s_kd_T2_a05")}
out = []
for name, m in models.items():
    b1 = robust_ms(torch_fn(m, 1))
    b32 = robust_ms(torch_fn(m, 32), blocks=3, runs=8, warmup=3)
    out.append(dict(model=name, b1_best=b1[0], b1_typ=b1[1], b1_worst=b1[2],
                    b32_best=b32[0], b32_typ=b32[1], b32_worst=b32[2]))
LAT = pd.DataFrame(out); LAT["b1_spread"] = (LAT.b1_worst / LAT.b1_best).round(2)
LAT.to_csv("results/latency_robust.csv", index=False)
print(LAT.to_string(index=False))

           model  b1_best  b1_typ  b1_worst  b32_best  b32_typ  b32_worst  b1_spread
B0 baseline FP32     39.2    39.8     133.3    1452.1   1469.1     1676.8       3.40
        mnv3s_ce     13.4    13.6      13.7     170.0    173.3      175.2       1.02
 mnv3s_kd_T2_a05     13.2    13.6      38.0     167.9    168.2      169.8       2.88


In [14]:
print("TAG:", TAG)
print("rows in eff:", eff.model.tolist())

TAG: mnv3s_kd_T2_a05
rows in eff: ['B0 baseline FP32', 'mnv3s_ce', 'mnv3s_kd_T2_a05']


In [15]:
%%writefile src/onnx_tools.py
import time, numpy as np, onnxruntime as ort
from engine import prep, make_loader

def ort_session(path, threads=2):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

def ort_scores(path, ds):
    s, out = ort_session(path), []
    for x, _, _ in make_loader(ds, False):
        z = s.run(None, {"mel": prep(x).numpy()})[0]
        out.append(1 / (1 + np.exp(-z)))
    return np.concatenate(out)

def robust_ms(fn, blocks=5, runs=40, warmup=10):
    for _ in range(warmup): fn()
    meds = []
    for _ in range(blocks):
        ts = []
        for _ in range(runs):
            t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
        meds.append(np.median(ts) * 1000)
    return round(min(meds), 1), round(float(np.median(meds)), 1), round(max(meds), 1)

def ort_latency(path, batch, runs):
    s = ort_session(path); x = np.random.randn(batch, 1, 128, 313).astype(np.float32)
    return robust_ms(lambda: s.run(None, {"mel": x}), blocks=5, runs=max(4, runs // 2))[0]

Overwriting src/onnx_tools.py


In [16]:
!git add src/onnx_tools.py && git commit -qm "Add ONNX scoring and timing helpers" && git push -q

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	results/distill_seeds.csv
	results/efficiency.csv
	results/latency_robust.csv
	results/mnv3s_ce_history.csv
	results/mnv3s_kd_T2_a05_history.csv

nothing added to commit but untracked files present (use "git add" to track)


In [17]:
import os, shutil, numpy as np, pandas as pd, torch
!pip install -q onnx onnxruntime
import onnx, onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader
from onnxruntime.quantization.shape_inference import quant_pre_process
from onnx_tools import ort_session, ort_scores, ort_latency, robust_ms
from profile_tools import mel_ms, log_row, count_params, gflops
from engine import prep

TAG = "mnv3s_kd_T2_a05"
EXP, DRV = "/content/export", "/content/drive/MyDrive/birdclef_project/export"
os.makedirs(EXP, exist_ok=True); os.makedirs(DRV, exist_ok=True)
FP32, PRE = f"{EXP}/student_fp32.onnx", f"{EXP}/student_pre.onnx"

def load_student(tag):
    m = make_model("mobilenetv3_small_100", pretrained=False)
    m.load_state_dict(torch.load(f"{CK}/{tag}_best.pt", map_location="cpu")); return m.eval()

student = load_student(TAG)
if os.path.exists(f"{DRV}/student_fp32.onnx"):
    shutil.copy(f"{DRV}/student_fp32.onnx", FP32); print("restored FP32 ONNX from Drive")
else:
    torch.onnx.export(student, torch.randn(1, 1, 128, 313), FP32, input_names=["mel"], output_names=["logits"],
                      dynamic_axes={"mel": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)
    shutil.copy(FP32, DRV); print("re-exported FP32 ONNX from the checkpoint and backed it up")

try:
    quant_pre_process(FP32, PRE); src = PRE
except Exception as e:
    print("pre-processing skipped:", e); src = FP32

rng = np.random.RandomState(0)
cal_rows = rng.choice(len(rows["train"]), 320, replace=False, p=w / w.sum())
mm = np.load(ALL, mmap_mode="r")
batches = []
for s in range(0, len(cal_rows), 16):
    xb = torch.from_numpy(np.stack([np.array(mm[rows["train"][i]], dtype=np.float32) for i in cal_rows[s:s+16]]))
    batches.append({"mel": prep(xb).numpy()})

class Reader(CalibrationDataReader):
    def __init__(self, b): self.it = iter(b)
    def get_next(self): return next(self.it, None)

va32 = macro_auc(Y[rows["val"]], ort_scores(FP32, val_ds))[0]
print(f"FP32 ONNX val {va32:.4f}  (should be 0.9611) | calibration batches: {len(batches)}")

restored FP32 ONNX from Drive
FP32 ONNX val 0.9611  (should be 0.9611) | calibration batches: 20


In [18]:
from onnxruntime.quantization import CalibrationMethod

va32, _ = macro_auc(Y[rows["val"]], ort_scores(FP32, val_ds))
print(f"FP32 ONNX, val {va32:.4f}   (PyTorch student: 0.9611)")

def quantize_variant(dst, calib, op_types=None):
    quantize_static(src, dst, Reader(batches), quant_format=QuantFormat.QDQ,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8,
                    per_channel=True, calibrate_method=calib, op_types_to_quantize=op_types)

variants = {"B": ("percentile, all ops", CalibrationMethod.Percentile, None),
            "C": ("entropy, all ops",    CalibrationMethod.Entropy,    None),
            "D": ("minmax, Conv only",   CalibrationMethod.MinMax,     ["Conv"]),
            "E": ("percentile, Conv only", CalibrationMethod.Percentile, ["Conv"])}
rows_out = [dict(key="A", variant="minmax, all ops (first attempt)", size_MB=round(os.path.getsize(INT8) / 1e6, 2), val_auc=0.7907)]
for key, (label, calib, ops) in variants.items():
    path = f"/content/export/student_int8_{key}.onnx"
    try:
        quantize_variant(path, calib, ops)
        v, _ = macro_auc(Y[rows["val"]], ort_scores(path, val_ds))
        rows_out.append(dict(key=key, variant=label, size_MB=round(os.path.getsize(path) / 1e6, 2), val_auc=round(v, 4)))
    except Exception as e:
        rows_out.append(dict(key=key, variant=label, size_MB=np.nan, val_auc=np.nan))
        print(f"variant {key} failed: {type(e).__name__}: {str(e)[:150]}")
    print(rows_out[-1])
print(pd.DataFrame(rows_out).to_string(index=False))

FP32 ONNX, val 0.9611   (PyTorch student: 0.9611)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 142
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
{'key': 'B', 'variant': 'percentile, all ops', 'size_MB': 2.04, 'val_auc': 0.8349}
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 142
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
{'key': 'C', 'variant': 'entropy, all ops', 'size_MB': 2.04, 'val_auc': 0.7907}
{'key': 'D', 'variant': 'minmax, Conv only', 'size_MB': 2.58, 'val_auc': 0.813}
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 101
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
{'key': 'E', 'variant': 'percentile, Conv only', 'size_MB': 2.58, 'val_auc': 0.8364}
key                         variant  size_MB  val_auc
  

In [19]:
from profile_tools import mel_ms, log_row, count_params, gflops

BEST_KEY = "D"      # <- the variant with the highest val AUC in the table
BEST = f"/content/export/student_int8_{BEST_KEY}.onnx"
mel = mel_ms()
params, gf = round(count_params(student) / 1e6, 3), round(gflops(student), 3)

todo = [("student FP32 (ONNX)", FP32), (f"student INT8-{BEST_KEY} (ONNX)", BEST)]
for name, path in todo:
    va = macro_auc(Y[rows["val"]], ort_scores(path, val_ds))[0]
    te = macro_auc(Y[rows["test"]], ort_scores(path, test_ds))[0]
    l1, l32 = ort_latency(path, 1, 100), ort_latency(path, 32, 20)
    log_row(dict(model=name, params_M=params, size_MB=round(os.path.getsize(path) / 1e6, 2), gflops=gf,
                 lat_b1_ms=round(l1, 1), lat_b32_ms=round(l32, 1), mel_ms=round(mel, 1), threads=2,
                 win_per_s_b1=round(1000 / (mel + l1), 1), win_per_s_b32=round(32 * 1000 / (32 * mel + l32), 1),
                 val_auc=round(va, 4), test_auc=round(te, 4)))
    print(name, "| val", round(va, 4), "| test", round(te, 4))
print(pd.read_csv("results/efficiency.csv")[["model", "size_MB", "lat_b1_ms", "lat_b32_ms", "mel_ms",
      "win_per_s_b1", "win_per_s_b32", "val_auc", "test_auc"]].to_string(index=False))

student FP32 (ONNX) | val 0.9611 | test 0.9634
student INT8-D (ONNX) | val 0.813 | test 0.8114
                model  size_MB  lat_b1_ms  lat_b32_ms  mel_ms  win_per_s_b1  win_per_s_b32  val_auc  test_auc
     B0 baseline FP32    17.25       38.4      1390.9     7.7          21.7           19.5   0.9627    0.9653
             mnv3s_ce     6.95       14.1       556.4    15.1          34.2           30.7   0.9567    0.9556
      mnv3s_kd_T2_a05     6.95       13.7       199.5     7.7          46.6           71.5   0.9611    0.9634
  student FP32 (ONNX)     6.82        4.7        62.9     7.8          80.0          102.4   0.9611    0.9634
student INT8-D (ONNX)     2.58       11.0       125.0     7.8          53.2           85.4   0.8130    0.8114


In [20]:
import subprocess, onnx
import onnx, onnxruntime as ort, time
from onnxruntime.quantization import (quantize_static, QuantType, QuantFormat, CalibrationMethod)

lscpu = subprocess.run("lscpu", shell=True, capture_output=True, text=True).stdout.splitlines()
print([l.strip() for l in lscpu if "Model name" in l])
flags = set(next((l for l in lscpu if l.startswith("Flags")), "").split())
print({f: f in flags for f in ["avx2", "avx512f", "avx512_vnni", "avx_vnni", "amx_int8"]})

mdl = onnx.load(src)
convs = [n for n in mdl.graph.node if n.op_type == "Conv"]
first, last = convs[0].name, convs[-1].name
se = [n.name for n in convs if "conv_reduce" in n.name or "conv_expand" in n.name or "/se/" in n.name]
print("conv nodes:", len(convs), "| SE convs matched:", len(se), "| first:", first, "| last:", last)

def qv(dst, ops=("Conv",), reduce_range=False, wtype=QuantType.QInt8, exclude=None):
    quantize_static(src, dst, Reader(batches), quant_format=QuantFormat.QDQ,
                    activation_type=QuantType.QUInt8, weight_type=wtype, per_channel=True,
                    reduce_range=reduce_range, calibrate_method=CalibrationMethod.Percentile,
                    op_types_to_quantize=list(ops) if ops else None, nodes_to_exclude=exclude or [])

V = {"F": ("all ops + reduce_range",              dict(ops=None, reduce_range=True)),
     "G": ("Conv only + reduce_range",            dict(reduce_range=True)),
     "H": ("Conv only, U8U8 weights",             dict(wtype=QuantType.QUInt8)),
     "I": ("Conv only + reduce_range, keep first/last/SE convs in float",
           dict(reduce_range=True, exclude=[first, last] + se))}
out = []
for k, (label, kw) in V.items():
    p = f"/content/export/student_int8_{k}.onnx"
    try:
        qv(p, **kw)
        v = macro_auc(Y[rows["val"]], ort_scores(p, val_ds))[0]
        out.append(dict(key=k, variant=label, size_MB=round(os.path.getsize(p) / 1e6, 2), val_auc=round(v, 4)))
    except Exception as e:
        out.append(dict(key=k, variant=label, size_MB=np.nan, val_auc=np.nan))
        print(f"{k} failed: {type(e).__name__}: {str(e)[:150]}")
    print(out[-1])
print(pd.DataFrame(out).to_string(index=False), "\n(reference: E 0.8364, FP32 0.9611)")

['Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz']
{'avx2': True, 'avx512f': True, 'avx512_vnni': False, 'avx_vnni': False, 'amx_int8': False}
conv nodes: 53 | SE convs matched: 18 | first: /conv_stem/Conv | last: /conv_head/Conv
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 142
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
{'key': 'F', 'variant': 'all ops + reduce_range', 'size_MB': 2.04, 'val_auc': 0.8339}
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 101
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
{'key': 'G', 'variant': 'Conv only + reduce_range', 'size_MB': 2.58, 'val_auc': 0.8363}
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 101
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)


{'key': 'H', 'variant': 'Conv only, U8U8 weights', 'size_MB': 2.59, 'val_auc': 0.8339}
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 101
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
{'key': 'I', 'variant': 'Conv only + reduce_range, keep first/last/SE convs in float', 'size_MB': 5.63, 'val_auc': 0.8244}
key                                                     variant  size_MB  val_auc
  F                                      all ops + reduce_range     2.04   0.8339
  G                                    Conv only + reduce_range     2.58   0.8363
  H                                     Conv only, U8U8 weights     2.59   0.8339
  I Conv only + reduce_range, keep first/last/SE convs in float     5.63   0.8244 
(reference: E 0.8364, FP32 0.9611)


In [21]:
import copy
mel = mel_ms()
B0 = "/content/export/b0_fp32.onnx"
b0m = copy.deepcopy(teacher).cpu().eval()
torch.onnx.export(b0m, torch.randn(1, 1, 128, 313), B0, input_names=["mel"], output_names=["logits"],
                  dynamic_axes={"mel": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)
xt = torch.randn(8, 1, 128, 313)
with torch.no_grad(): ref = b0m(xt).numpy()
print("max |torch - onnx|:", float(np.abs(ref - ort_session(B0).run(None, {"mel": xt.numpy()})[0]).max()))

eff = pd.read_csv("results/efficiency.csv")
b0row = eff[eff.model.str.startswith("B0")]
assert len(b0row), "B0 row missing from efficiency.csv: re-run Block 5.3"
b0row = b0row.iloc[0]
va = macro_auc(Y[rows["val"]], ort_scores(B0, val_ds))[0]
te = macro_auc(Y[rows["test"]], ort_scores(B0, test_ds))[0]
l1, l32 = ort_latency(B0, 1, 100), ort_latency(B0, 32, 20)
log_row(dict(model="B0 FP32 (ONNX)", params_M=b0row.params_M, size_MB=round(os.path.getsize(B0) / 1e6, 2),
             gflops=b0row.gflops, lat_b1_ms=round(l1, 1), lat_b32_ms=round(l32, 1), mel_ms=round(mel, 1), threads=2,
             win_per_s_b1=round(1000 / (mel + l1), 1), win_per_s_b32=round(32 * 1000 / (32 * mel + l32), 1),
             val_auc=round(va, 4), test_auc=round(te, 4)))
eff = pd.read_csv("results/efficiency.csv")
print(eff[eff.model.str.contains("ONNX")][["model", "size_MB", "lat_b1_ms", "lat_b32_ms", "win_per_s_b1", "val_auc", "test_auc"]].to_string(index=False))

/tmp/ipykernel_2103/1995632427.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(b0m, torch.randn(1, 1, 128, 313), B0, input_names=["mel"], output_names=["logits"],


max |torch - onnx|: 1.33514404296875e-05
                model  size_MB  lat_b1_ms  lat_b32_ms  win_per_s_b1  val_auc  test_auc
  student FP32 (ONNX)     6.82        4.7        62.9          80.0   0.9611    0.9634
student INT8-D (ONNX)     2.58       11.0       125.0          53.2   0.8130    0.8114
       B0 FP32 (ONNX)    16.94       13.2       502.5          47.5   0.9627    0.9653


In [22]:
from metrics import per_class_auc
n_train = pd.Series(idx.primary_label.values[rows["train"]]).value_counts().reindex(CLASSES).values
rare = n_train <= np.quantile(n_train, 0.25)
mods = {"teacher B0": teacher, "student control": load_student("mnv3s_ce"), "student distilled": load_student("mnv3s_kd_T2_a05")}
A, rep = {}, []
for name, m in mods.items():
    m = m.to(DEV).eval()
    for split, ds in [("val", val_ds), ("test", test_ds)]:
        a = per_class_auc(Y[rows[split]], predict(m, make_loader(ds, False)))
        A[(name, split)] = a
        rep.append(dict(model=name, split=split, macro=np.nanmean(a), rare=np.nanmean(a[rare]), common=np.nanmean(a[~rare])))
print(pd.DataFrame(rep).round(4).to_string(index=False))
for split in ["val", "test"]:
    d, c = A[("student distilled", split)], A[("student control", split)]
    ok = rare & ~np.isnan(d) & ~np.isnan(c)
    print(f"{split}: distilled beats control on {int((d[ok] > c[ok]).sum())} of {int(ok.sum())} rare species")

            model split  macro   rare  common
       teacher B0   val 0.9627 0.9537  0.9658
       teacher B0  test 0.9653 0.9575  0.9679
  student control   val 0.9567 0.9544  0.9575
  student control  test 0.9556 0.9405  0.9608
student distilled   val 0.9611 0.9551  0.9632
student distilled  test 0.9634 0.9592  0.9648
val: distilled beats control on 29 of 47 rare species
test: distilled beats control on 32 of 47 rare species


In [ ]:
from onnx import numpy_helper

def weight_only(src_path, dst_path, bits):
    m = onnx.load(src_path)
    conv_w = {n.input[1] for n in m.graph.node if n.op_type == "Conv"}
    qmax = 2 ** (bits - 1) - 1
    for init in m.graph.initializer:
        if init.name in conv_w:
            W = numpy_helper.to_array(init).astype(np.float32)
            ax = tuple(range(1, W.ndim))
            sc = np.maximum(np.abs(W).max(axis=ax, keepdims=True), 1e-8) / qmax
            Wq = np.clip(np.round(W / sc), -qmax, qmax) * sc
            init.CopyFrom(numpy_helper.from_array(Wq.astype(np.float32), init.name))
    onnx.save(m, dst_path)

for b in [8, 6, 4]:
    p = f"{EXP}/student_w{b}.onnx"; weight_only(FP32, p, b)
    v = macro_auc(Y[rows["val"]], ort_scores(p, val_ds))[0]
    print(f"conv weights rounded to {b} bits, activations in float: val {v:.4f}   (FP32: 0.9611)")

conv weights rounded to 8 bits, activations in float: val 0.9611   (FP32: 0.9611)
conv weights rounded to 6 bits, activations in float: val 0.9592   (FP32: 0.9611)


In [ ]:
import json, matplotlib.pyplot as plt
eff = pd.read_csv("results/efficiency.csv")
E = eff[eff.model.str.contains("ONNX")].copy()
b0 = E[E.model.str.startswith("B0")].iloc[0]
E["size_x_smaller"] = (b0.size_MB / E.size_MB).round(2)
E["b1_x_faster"]    = (b0.lat_b1_ms / E.lat_b1_ms).round(2)
E["b32_x_faster"]   = (b0.lat_b32_ms / E.lat_b32_ms).round(2)
E["e2e_x_faster"]   = (E.win_per_s_b1 / b0.win_per_s_b1).round(2)
cols = ["model", "size_MB", "lat_b1_ms", "lat_b32_ms", "win_per_s_b1", "val_auc", "test_auc",
        "size_x_smaller", "b1_x_faster", "b32_x_faster", "e2e_x_faster"]
print(E[cols].to_string(index=False)); E[cols].to_csv("results/final_compression_table.csv", index=False)

d = pd.read_csv("results/distill_seeds.csv")
base = json.load(open("results/baseline_metrics.json"))
abl = d.groupby("model")[["val_auc", "test_auc"]].mean().round(4)
abl.loc["teacher B0 (1 run)"] = [round(base["val_auc"], 4), round(base["test_auc"], 4)]
print("\ntraining ablation (mean of 2 seeds):\n", abl)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for _, r in E.iterrows():
    nm = r.model.replace(" (ONNX)", "")
    for a, x in [(ax[0], r.size_MB), (ax[1], r.win_per_s_b1)]:
        a.scatter(x, r.test_auc, s=70); a.annotate(nm, (x, r.test_auc), textcoords="offset points", xytext=(6, 6), fontsize=8)
ax[0].set_xlabel("model size (MB)"); ax[1].set_xlabel("end-to-end windows/s (batch 1, incl. mel, 2 CPU threads)")
for a in ax: a.set_ylabel("test macro-AUC"); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/pareto.png", dpi=130); plt.show()

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/05_compression.ipynb" notebooks/
!git add -A
!git status --short
!git commit -m "Phase 5 complete: final compression table, Pareto plot, INT8 negative result"
!git push